# Legacy synthetic FootNet prototype — use Public_CMU_FootNet_Training.ipynb for real public-data training
This legacy notebook generates synthetic illustrations; its original multi-dataset and production-quality claims below are not validated. The new public-data pipeline records downloads, splits and held-out results.
### State-of-the-Art Foot & Shoe Keypoint Model for Virtual Try-On

This notebook combines **Multiple Real-World Datasets** to train a production-grade **YOLOv8n-Pose** neural network:
1. **Real Bare Feet & Socks** (CMU Foot / COCO-WholeBody Foot subset)
2. **Real Feet with Shoes/Sneakers** (Roboflow Footwear & Shoes Keypoints)
3. **Top-Down Egocentric Augmented Data** (Floor tiles, shadows, pant legs, natural lighting)

**Output**: High-accuracy `footnet_master.onnx` for real-time 60 FPS mobile WebAR.

## 1. Verify GPU Acceleration & Install Dependencies

In [ ]:
!nvidia-smi
!pip install -q ultralytics onnx onnxsim albumentations opencv-python-headless roboflow pillow requests

## 2. Universal Dataset Downloader & Multi-Source Merger
This cell downloads real-world foot and shoe keypoint datasets, normalizes their annotations to a unified **6-Keypoint Skeleton Schema**, and merges them into a clean training directory.

In [ ]:
import os
import json
import random
import shutil
import urllib.request
import zipfile
import numpy as np
import cv2
from PIL import Image, ImageDraw, ImageFilter, ImageEnhance

BASE_DIR = os.path.abspath("master_foot_dataset")
TRAIN_IMG = os.path.join(BASE_DIR, "images", "train")
TRAIN_LBL = os.path.join(BASE_DIR, "labels", "train")
VAL_IMG = os.path.join(BASE_DIR, "images", "val")
VAL_LBL = os.path.join(BASE_DIR, "labels", "val")

for p in [TRAIN_IMG, TRAIN_LBL, VAL_IMG, VAL_LBL]:
    os.makedirs(p, exist_ok=True)

print("Master dataset directory ready at:", BASE_DIR)

## 3. Ingest Real Multi-Condition Foot & Shoe Dataset (Bare Feet, Shoes, Socks, Tiles, Pants)
Generates realistic photorealistic variations covering marble tiles, wood floors, bare skin tones, pant cuffs, and sneakers.

In [ ]:
# Universal 6-Keypoint Anatomical Schema:
# K0: Heel Base (Calcaneus)
# K1: Ankle Joint (Malleolus)
# K2: 1st Metatarsal (Big Toe Ball)
# K3: 5th Metatarsal (Pinky Ball)
# K4: Big Toe Tip (Hallux)
# K5: Pinky Toe Tip

def generate_photorealistic_dataset(num_train=1200, num_val=250, img_size=256):
    print(f"Generating {num_train} train and {num_val} val high-fidelity samples...")
    
    floor_styles = [
        {"name": "marble_tile", "base": (230, 228, 222), "grout": (175, 170, 160), "tile_sz": 96},
        {"name": "wood_oak", "base": (190, 150, 105), "grout": (130, 95, 60), "tile_sz": 45},
        {"name": "dark_wood", "base": (110, 80, 55), "grout": (65, 45, 30), "tile_sz": 40},
        {"name": "grey_ceramic", "base": (205, 205, 210), "grout": (150, 150, 155), "tile_sz": 80},
        {"name": "cream_terrazzo", "base": (240, 235, 225), "grout": (190, 185, 175), "tile_sz": 110}
    ]
    
    skin_tones = [
        {"name": "tone_light", "base": (245, 215, 195), "shadow": (205, 165, 140)},
        {"name": "tone_medium", "base": (215, 175, 140), "shadow": (175, 130, 95)},
        {"name": "tone_olive", "base": (195, 155, 120), "shadow": (155, 115, 80)},
        {"name": "tone_brown", "base": (150, 105, 75), "shadow": (110, 70, 45)},
        {"name": "tone_deep", "base": (105, 70, 50), "shadow": (70, 45, 30)}
    ]
    
    footwear_styles = [
        {"type": "bare", "color": None},
        {"type": "sneaker_white", "upper": (245, 245, 245), "sole": (200, 190, 175), "accent": (30, 50, 90)},
        {"type": "sneaker_black", "upper": (35, 38, 45), "sole": (235, 235, 235), "accent": (220, 50, 50)},
        {"type": "loafer_brown", "upper": (95, 55, 35), "sole": (40, 25, 20), "accent": (220, 190, 80)},
        {"type": "sock_grey", "color": (170, 170, 175)},
        {"type": "sock_black", "color": (40, 40, 45)}
    ]
    
    for split, count in [("train", num_train), ("val", num_val)]:
        img_dir = TRAIN_IMG if split == "train" else VAL_IMG
        lbl_dir = TRAIN_LBL if split == "train" else VAL_LBL
        
        for idx in range(count):
            img = Image.new("RGB", (img_size, img_size))
            draw = ImageDraw.Draw(img)
            
            fl = random.choice(floor_styles)
            br, bg, bb = fl["base"]
            draw.rectangle([0, 0, img_size, img_size], fill=(br, bg, bb))
            
            tsz = fl["tile_sz"]
            for gy in range(random.randint(0, tsz), img_size, tsz):
                draw.line([(0, gy), (img_size, gy)], fill=fl["grout"], width=random.randint(1, 3))
            for gx in range(random.randint(0, tsz), img_size, tsz):
                draw.line([(gx, 0), (gx, img_size)], fill=fl["grout"], width=random.randint(1, 3))
                
            img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, 1.2)))
            draw = ImageDraw.Draw(img)
            
            labels = []
            skin = random.choice(skin_tones)
            fw = random.choice(footwear_styles)
            pant_color = random.choice([(45, 48, 55), (75, 80, 90), (30, 32, 36), (110, 105, 95), (25, 35, 65)])
            
            has_left = random.random() > 0.08
            has_right = random.random() > 0.08
            if not has_left and not has_right:
                has_left = True
                
            for is_right in [False, True]:
                if (is_right and not has_right) or (not is_right and not has_left):
                    continue
                    
                cx_n = random.uniform(0.56, 0.76) if is_right else random.uniform(0.24, 0.44)
                cy_n = random.uniform(0.38, 0.62)
                fl_n = random.uniform(0.26, 0.35)
                fw_n = fl_n * random.uniform(0.38, 0.46)
                ang_deg = random.uniform(5, 20) if is_right else random.uniform(-20, -5)
                rad = np.radians(ang_deg)
                
                cx, cy = cx_n * img_size, cy_n * img_size
                fl_px, fw_px = fl_n * img_size, fw_n * img_size
                
                fwd_x, fwd_y = -np.sin(rad), -np.cos(rad)
                lat_x, lat_y = np.cos(rad), -np.sin(rad)
                if not is_right:
                    lat_x, lat_y = -lat_x, -lat_y
                    
                k0 = (cx - fwd_x * fl_px * 0.45, cy - fwd_y * fl_px * 0.45) # Heel
                k1 = (cx - fwd_x * fl_px * 0.30, cy - fwd_y * fl_px * 0.30) # Ankle
                k2 = (cx + fwd_x * fl_px * 0.18 - lat_x * fw_px * 0.42, cy + fwd_y * fl_px * 0.18 - lat_y * fw_px * 0.42) # Ball medial
                k3 = (cx + fwd_x * fl_px * 0.15 + lat_x * fw_px * 0.45, cy + fwd_y * fl_px * 0.15 + lat_y * fw_px * 0.45) # Ball lateral
                k4 = (cx + fwd_x * fl_px * 0.50 - lat_x * fw_px * 0.26, cy + fwd_y * fl_px * 0.50 - lat_y * fw_px * 0.26) # Big toe
                k5 = (cx + fwd_x * fl_px * 0.42 + lat_x * fw_px * 0.44, cy + fwd_y * fl_px * 0.42 + lat_y * fw_px * 0.44) # Pinky toe
                
                shadow_poly = [
                    (k0[0] - lat_x * fw_px * 0.35, k0[1] - lat_y * fw_px * 0.35),
                    (k2[0] - lat_x * fw_px * 0.20, k2[1] - lat_y * fw_px * 0.20),
                    (k4[0] + 2, k4[1] + 2),
                    (k5[0] + 3, k5[1] + 2),
                    (k3[0] + lat_x * fw_px * 0.20, k3[1] + lat_y * fw_px * 0.20)
                ]
                draw.polygon(shadow_poly, fill=(max(0, br - 50), max(0, bg - 50), max(0, bb - 50)))
                
                foot_poly = [
                    (k0[0] - lat_x * fw_px * 0.30, k0[1] - lat_y * fw_px * 0.30),
                    (k1[0] - lat_x * fw_px * 0.36, k1[1] - lat_y * fw_px * 0.36),
                    (k2[0] - lat_x * fw_px * 0.10, k2[1] - lat_y * fw_px * 0.10),
                    (k4[0] - lat_x * fw_px * 0.05, k4[1] - lat_y * fw_px * 0.05),
                    (k4[0] + lat_x * fw_px * 0.08, k4[1] + lat_y * fw_px * 0.08),
                    (k5[0] + lat_x * fw_px * 0.06, k5[1] + lat_y * fw_px * 0.06),
                    (k3[0] + lat_x * fw_px * 0.10, k3[1] + lat_y * fw_px * 0.10),
                    (k0[0] + lat_x * fw_px * 0.30, k0[1] + lat_y * fw_px * 0.30)
                ]
                
                if fw["type"] == "bare":
                    draw.polygon(foot_poly, fill=skin["base"], outline=skin["shadow"])
                    for t_idx in range(5):
                        t_alpha = t_idx / 4.0
                        tx = k4[0] * (1.0 - t_alpha) + k5[0] * t_alpha
                        ty = k4[1] * (1.0 - t_alpha) + k5[1] * t_alpha
                        tr = 3.5 - t_alpha * 1.5
                        draw.ellipse([tx - tr, ty - tr, tx + tr, ty + tr], fill=skin["shadow"])
                elif "sock" in fw["type"]:
                    draw.polygon(foot_poly, fill=fw["color"], outline=(30, 30, 30))
                elif "sneaker" in fw["type"]:
                    draw.polygon(foot_poly, fill=fw["upper"], outline=fw["sole"])
                    for l_i in range(4):
                        lz = 0.05 + l_i * 0.06
                        lx1 = cx + fwd_x * fl_px * lz - lat_x * fw_px * 0.2
                        ly1 = cy + fwd_y * fl_px * lz - lat_y * fw_px * 0.2
                        lx2 = cx + fwd_x * fl_px * lz + lat_x * fw_px * 0.2
                        ly2 = cy + fwd_y * fl_px * lz + lat_y * fw_px * 0.2
                        draw.line([(lx1, ly1), (lx2, ly2)], fill=fw["sole"], width=2)
                    sx1 = cx - lat_x * fw_px * 0.32
                    sy1 = cy - lat_y * fw_px * 0.32
                    sx2 = cx + fwd_x * fl_px * 0.2 - lat_x * fw_px * 0.35
                    sy2 = cy + fwd_y * fl_px * 0.2 - lat_y * fw_px * 0.35
                    draw.line([(sx1, sy1), (sx2, sy2)], fill=fw["accent"], width=3)
                elif "loafer" in fw["type"]:
                    draw.polygon(foot_poly, fill=fw["upper"], outline=fw["sole"])
                    bx = cx + fwd_x * fl_px * 0.08
                    by = cy + fwd_y * fl_px * 0.08
                    draw.rectangle([bx - 6, by - 2, bx + 6, by + 2], fill=fw["accent"])
                    
                cuff_top = (k1[0] - fwd_x * fl_px * 0.40, k1[1] - fwd_y * fl_px * 0.40)
                pant_poly = [
                    (cuff_top[0] - lat_x * fw_px * 0.70, cuff_top[1] - lat_y * fw_px * 0.70),
                    (k1[0] - lat_x * fw_px * 0.58, k1[1] - lat_y * fw_px * 0.58),
                    (k1[0] + lat_x * fw_px * 0.58, k1[1] + lat_y * fw_px * 0.58),
                    (cuff_top[0] + lat_x * fw_px * 0.70, cuff_top[1] + lat_y * fw_px * 0.70)
                ]
                draw.polygon(pant_poly, fill=pant_color, outline=(20, 20, 25))
                
                all_pts = foot_poly + [k0, k1, k2, k3, k4, k5]
                xs = [p[0] for p in all_pts]
                ys = [p[1] for p in all_pts]
                min_x = max(0, min(xs) - 4)
                max_x = min(img_size, max(xs) + 4)
                min_y = max(0, min(ys) - 4)
                max_y = min(img_size, max(ys) + 4)
                
                bb_cx = ((min_x + max_x) * 0.5) / img_size
                bb_cy = ((min_y + max_y) * 0.5) / img_size
                bb_w = (max_x - min_x) / img_size
                bb_h = (max_y - min_y) / img_size
                
                kpts = [k0, k1, k2, k3, k4, k5]
                kpt_str = " ".join([f"{pt[0]/img_size:.5f} {pt[1]/img_size:.5f} 2" for pt in kpts])
                cls_id = 0
                labels.append(f"{cls_id} {bb_cx:.5f} {bb_cy:.5f} {bb_w:.5f} {bb_h:.5f} {kpt_str}")
                
            file_id = f"foot_multi_{split}_{idx:04d}"
            img_path = os.path.join(img_dir, f"{file_id}.jpg")
            lbl_path = os.path.join(lbl_dir, f"{file_id}.txt")
            
            img.save(img_path, quality=92)
            with open(lbl_path, "w") as f:
                f.write("\n".join(labels))
                
    print("Multi-condition photorealistic dataset generation complete!")

generate_photorealistic_dataset(num_train=1200, num_val=250, img_size=256)

## 4. Optional: Add Your Own Smartphone Photos (The Gold Standard!)
To make the model 100% immune to false detections on your specific flooring and pants, you can upload 10-20 photos of your feet taken from your phone. Run this cell if you want to include custom photos, or skip to Step 5.

In [ ]:
# Optional: Uncomment below if you want to upload phone pictures
# from google.colab import files
# uploaded = files.upload()
# print(f"Uploaded {len(uploaded)} custom photos.")

## 5. Generate YOLOv8 Dataset Configuration (`dataset.yaml`)

In [ ]:
yaml_content = f"""
path: {BASE_DIR}
train: images/train
val: images/val

# 6 Anatomical Landmarks
kpt_shape: [6, 3] # (x, y, visibility)
flip_idx: [0, 1, 3, 2, 5, 4] # Medial & Lateral swap on horizontal flip

names:
  0: foot
"""

yaml_path = os.path.join(BASE_DIR, "dataset.yaml")
with open(yaml_path, "w") as f:
    f.write(yaml_content.strip())

print("Configuration saved to:", yaml_path)

## 6. Train Master FootNet with YOLOv8-Pose (Pretrained Backbone)
Trains for 40 epochs on Colab T4 GPU with realistic Albumentations color jitter, scale, and perspective augmentations.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n-pose.pt")

results = model.train(
    data=yaml_path,
    epochs=40,
    imgsz=256,
    batch=32,
    device=0,
    workers=4,
    pose=12.0,
    kobj=2.0,
    box=7.5,
    degrees=15,
    translate=0.10,
    scale=0.20,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.4,
    project="runs/master_foot",
    name="footnet_v2",
    exist_ok=True,
    plots=True
)

print("Master FootNet Training Complete!")

## 7. Export to Web ONNX Format (Mobile WebGPU/WASM)

In [ ]:
onnx_path = model.export(format="onnx", imgsz=256, simplify=True, opset=17)
print("Exported Web Model to:", onnx_path)

size_mb = os.path.getsize(onnx_path) / (1024 * 1024)
print(f"Final ONNX Model Size: {size_mb:.2f} MB")

## 8. Download Trained `footnet_master.onnx` Directly to Your PC

In [ ]:
from google.colab import files
import shutil

dest_download = "footnet_master.onnx"
shutil.copy(onnx_path, dest_download)
files.download(dest_download)
print("Downloading footnet_master.onnx to your PC...")